Load the transformed data

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import os

In [2]:
spark = (
    SparkSession.builder
    .appName("SupportIQ-WindowFunctions")
    .master("local[*]")
    .getOrCreate()
)

In [3]:
transformed_path = "../data/processed/support_tickets_transformed"

In [4]:
parquet_files = [
    os.path.join(transformed_path, file)
    for file in os.listdir(transformed_path)
    if file.endswith(".parquet")
]

In [5]:
tickets_df = spark.read.parquet(
    *parquet_files
)

print("Data loaded successfully.")

Data loaded successfully.


In [6]:
tickets_df.show(5, truncate=False)

+----------+-----------+--------+------------+-----------+--------+---------------+-----------------+-------+-----------+-------+-------------------------------------------------+----------------------------------------------------+---------------+----------------------+----------------+---------------------+---------------+------------+-------------+-------------------+-----------------------+---------------------+--------------------+--------------+
|ticket_id |customer_id|agent_id|created_date|closed_date|priority|category       |sub_category     |product|status     |channel|issue_description                                |resolution                                          |customer_region|first_response_minutes|resolution_hours|customer_satisfaction|escalation_flag|created_year|created_month|resolution_category|first_response_category|satisfaction_category|escalation_indicator|priority_level|
+----------+-----------+--------+------------+-----------+--------+---------------+-----

Create a Customer Window

In [8]:
customer_window = Window.partitionBy(
    "customer_id"
).orderBy(
    F.col("created_date").desc()
)

Assign Row Numbers

In [9]:
tickets_with_row_number = tickets_df.withColumn(
    "row_number",
    F.row_number().over(customer_window)
)

In [10]:
tickets_with_row_number.select(
    "customer_id",
    "ticket_id",
    "created_date",
    "row_number"
).show(20, truncate=False)

+-----------+----------+------------+----------+
|customer_id|ticket_id |created_date|row_number|
+-----------+----------+------------+----------+
|CUST00005  |TKT0609868|2026-07-27  |1         |
|CUST00005  |TKT0210733|2026-06-08  |2         |
|CUST00005  |TKT0943682|2026-05-25  |3         |
|CUST00005  |TKT1145266|2026-03-30  |4         |
|CUST00005  |TKT0813843|2026-03-27  |5         |
|CUST00005  |TKT1120445|2026-03-15  |6         |
|CUST00005  |TKT0964387|2026-02-18  |7         |
|CUST00005  |TKT0043639|2026-02-15  |8         |
|CUST00005  |TKT0218692|2026-02-10  |9         |
|CUST00005  |TKT0029299|2026-01-15  |10        |
|CUST00005  |TKT0869684|2025-12-15  |11        |
|CUST00005  |TKT0042574|2025-11-27  |12        |
|CUST00005  |TKT0064996|2025-11-20  |13        |
|CUST00005  |TKT0982781|2025-11-14  |14        |
|CUST00005  |TKT1161711|2025-10-12  |15        |
|CUST00005  |TKT0089167|2025-09-17  |16        |
|CUST00005  |TKT0285008|2025-09-06  |17        |
|CUST00005  |TKT0436

Find the Latest Ticket for Each Customer

In [11]:
latest_ticket_per_customer = tickets_with_row_number.filter(
    F.col("row_number") == 1
)

In [12]:
latest_ticket_per_customer.select(
    "customer_id",
    "ticket_id",
    "created_date",
    "priority",
    "category"
).show(20, truncate=False)

+-----------+----------+------------+--------+---------------+
|customer_id|ticket_id |created_date|priority|category       |
+-----------+----------+------------+--------+---------------+
|CUST00005  |TKT0609868|2026-07-27  |Unknown |Time Management|
|CUST00036  |TKT0784770|2026-07-20  |Low     |Performance    |
|CUST00050  |TKT0699251|2026-07-25  |Low     |Recruitment    |
|CUST00051  |TKT0637321|2026-08-31  |High    |Recruitment    |
|CUST00069  |TKT0966001|2026-08-22  |Unknown |Core HR        |
|CUST00086  |TKT0708712|2026-08-08  |Low     |Recruitment    |
|CUST00087  |TKT0163955|2026-08-30  |Critical|Benefits       |
|CUST00091  |TKT0367542|2026-08-10  |High    |Payroll        |
|CUST00099  |TKT0847514|2026-08-29  |Unknown |Core HR        |
|CUST00100  |TKT0605115|2026-07-22  |High    |Benefits       |
|CUST00108  |TKT0137057|2026-08-09  |Medium  |Recruitment    |
|CUST00118  |TKT0270914|2026-08-17  |High    |Time Management|
|CUST00130  |TKT0011116|2026-07-31  |Medium  |Time Mana

Rank Tickets by Resolution Time

In [13]:
category_window = Window.partitionBy(
    "category"
).orderBy(
    F.col("resolution_hours").desc()
)

In [14]:
ranked_tickets = tickets_df.withColumn(
    "resolution_rank",
    F.rank().over(category_window)
)

In [15]:
ranked_tickets.select(
    "category",
    "ticket_id",
    "resolution_hours",
    "resolution_rank"
).show(20, truncate=False)

+--------+----------+----------------+---------------+
|category|ticket_id |resolution_hours|resolution_rank|
+--------+----------+----------------+---------------+
|Payroll |TKT0871894|200.0           |1              |
|Payroll |TKT0264453|200.0           |1              |
|Payroll |TKT0629472|200.0           |1              |
|Payroll |TKT0559568|200.0           |1              |
|Payroll |TKT0018731|199.99          |5              |
|Payroll |TKT0148734|199.99          |5              |
|Payroll |TKT0698520|199.99          |5              |
|Payroll |TKT0109089|199.99          |5              |
|Payroll |TKT0120835|199.98          |9              |
|Payroll |TKT1052163|199.98          |9              |
|Payroll |TKT1179947|199.98          |9              |
|Payroll |TKT0475086|199.98          |9              |
|Payroll |TKT0210855|199.98          |9              |
|Payroll |TKT0329817|199.98          |9              |
|Payroll |TKT0719569|199.98          |9              |
|Payroll |

Find the Longest-Resolution Ticket per Category

In [16]:
longest_ticket_per_category = ranked_tickets.filter(
    F.col("resolution_rank") == 1
)

In [17]:
longest_ticket_per_category.select(
    "category",
    "ticket_id",
    "resolution_hours",
    "priority"
).show(20, truncate=False)

+-----------+----------+----------------+--------+
|category   |ticket_id |resolution_hours|priority|
+-----------+----------+----------------+--------+
|Benefits   |TKT0114657|200.0           |Low     |
|Benefits   |TKT0276248|200.0           |Critical|
|Benefits   |TKT0060681|200.0           |Low     |
|Benefits   |TKT0197853|200.0           |Medium  |
|Core HR    |TKT0799889|200.0           |High    |
|Core HR    |TKT1164120|200.0           |Critical|
|Core HR    |TKT0603137|200.0           |Critical|
|Payroll    |TKT0871894|200.0           |Low     |
|Payroll    |TKT0264453|200.0           |High    |
|Payroll    |TKT0629472|200.0           |Critical|
|Payroll    |TKT0559568|200.0           |Critical|
|Performance|TKT0365598|200.0           |Low     |
|Performance|TKT0992282|200.0           |Critical|
|Performance|TKT0940340|200.0           |Low     |
|Performance|TKT0800765|200.0           |Critical|
|Recruitment|TKT0155260|200.0           |Medium  |
|Recruitment|TKT0653339|200.0  